In [0]:
-- Gold aggregation: daily store sales
CREATE OR REPLACE TABLE fk_ce_2026_07_1372.gold.agg_daily_store_sales AS
SELECT
    d.calendar_date AS order_date,
    s.store_id,
    s.city,
    COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0) AS delivered_orders,
    COALESCE(SUM(CASE WHEN f.order_status = 'CANCELLED' THEN 1 ELSE 0 END), 0) AS cancelled_orders,
    COALESCE(SUM(f.units), 0) AS units_sold,
    COALESCE(SUM(f.gross_amount), 0) AS gross_amount,
    COALESCE(SUM(f.discount_amount), 0) AS discount_amount,
    COALESCE(SUM(f.net_item_amount), 0) AS net_item_revenue,
    COALESCE(SUM(f.delivery_fee), 0) AS delivery_fee_revenue,
    COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' AND f.is_on_time = TRUE THEN 1 ELSE 0 END), 0) AS on_time_orders,
    CASE WHEN COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0) > 0
         THEN ROUND(
            100.0 * COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' AND f.is_on_time = TRUE THEN 1 ELSE 0 END), 0) /
            COALESCE(SUM(CASE WHEN f.order_status = 'DELIVERED' THEN 1 ELSE 0 END), 0), 2)
         ELSE NULL END AS on_time_rate_pct
FROM fk_ce_2026_07_1372.gold.dim_date d
CROSS JOIN fk_ce_2026_07_1372.gold.dim_store s
LEFT JOIN fk_ce_2026_07_1372.gold.fact_orders f
    ON f.order_date = d.calendar_date
    AND f.store_id = s.store_id
GROUP BY d.calendar_date, s.store_id, s.city
ORDER BY d.calendar_date, s.store_id;


In [0]:
-- KPI summary view for whole period
CREATE OR REPLACE VIEW fk_ce_2026_07_1372.gold.v_kpi_summary AS
SELECT
    COUNT_IF(order_status = 'DELIVERED') AS delivered_orders,
    COUNT_IF(order_status = 'CANCELLED') AS cancelled_orders,
    SUM(net_item_amount) AS net_item_revenue,
    CASE WHEN COUNT_IF(order_status = 'DELIVERED') > 0
         THEN ROUND(SUM(net_item_amount) / COUNT_IF(order_status = 'DELIVERED'), 2)
         ELSE NULL END AS avg_order_value,
    ROUND(100.0 * COUNT_IF(is_on_time = TRUE AND order_status = 'DELIVERED') / NULLIF(COUNT_IF(order_status = 'DELIVERED'), 0), 2) AS on_time_rate_pct,
    ROUND(100.0 * COUNT_IF(order_status = 'CANCELLED') / NULLIF(COUNT_IF(order_status = 'DELIVERED') + COUNT_IF(order_status = 'CANCELLED'), 0), 2) AS cancellation_rate_pct,
    COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END) AS active_customers,
    ROUND(100.0 * COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END) FILTER (WHERE delivered_count >= 2) /
        NULLIF(COUNT(DISTINCT CASE WHEN order_status = 'DELIVERED' THEN customer_id END), 0), 2) AS repeat_customer_rate_pct
FROM (
    SELECT *,
           COUNT_IF(order_status = 'DELIVERED') OVER (PARTITION BY customer_id) AS delivered_count
    FROM fk_ce_2026_07_1372.gold.fact_orders
);


In [0]:
-- Ops reconciliation results aligned to assessment checks
CREATE or replace table fk_ce_2026_07_1372.ops.reconciliation_results (
    run_ts TIMESTAMP,
    check_name STRING,
    source_value DECIMAL(18,2),
    target_value DECIMAL(18,2),
    status STRING
);

INSERT INTO fk_ce_2026_07_1372.ops.reconciliation_results
SELECT
    current_timestamp() AS run_ts,
    check_name,
    source_value,
    target_value,
    CASE WHEN ABS(source_value - target_value) < 0.01 THEN 'PASS' ELSE 'FAIL' END AS status
FROM (
    -- 1. ORDER_COUNT: Distinct standardized order_id in bronze.orders vs rows in silver.orders + silver.quarantine_orders
    SELECT 'ORDER_COUNT' AS check_name,
           CAST((SELECT COUNT(DISTINCT order_id) FROM fk_ce_2026_07_1372.bronze.orders) AS DECIMAL(18,2)) AS source_value,
           CAST((SELECT COUNT(*) FROM fk_ce_2026_07_1372.silver.orders) + (SELECT COUNT(*) FROM fk_ce_2026_07_1372.silver.quarantine_orders) AS DECIMAL(18,2)) AS target_value
    UNION ALL
    -- 2. LINE_COUNT: Rows in silver.order_items vs rows in gold.fact_order_line
    SELECT 'LINE_COUNT',
           CAST((SELECT COUNT(*) FROM fk_ce_2026_07_1372.silver.order_items) AS DECIMAL(18,2)),
           CAST((SELECT COUNT(*) FROM fk_ce_2026_07_1372.gold.fact_order_line) AS DECIMAL(18,2))
    UNION ALL
    -- 3. NET_REVENUE: Sum of net_amount in silver.order_items for DELIVERED orders vs sum of net_item_revenue in gold.agg_daily_store_sales
    SELECT 'NET_REVENUE',
           CAST((SELECT SUM(net_amount) FROM fk_ce_2026_07_1372.silver.order_items oi INNER JOIN fk_ce_2026_07_1372.silver.orders o ON oi.order_id = o.order_id WHERE o.order_status = 'DELIVERED') AS DECIMAL(18,2)),
           CAST((SELECT SUM(net_item_revenue) FROM fk_ce_2026_07_1372.gold.agg_daily_store_sales) AS DECIMAL(18,2))
    UNION ALL
    -- 4. DELIVERED_ORDERS: DELIVERED rows in gold.fact_orders vs sum of delivered_orders in gold.agg_daily_store_sales
    SELECT 'DELIVERED_ORDERS',
           CAST((SELECT COUNT(*) FROM fk_ce_2026_07_1372.gold.fact_orders WHERE order_status = 'DELIVERED') AS DECIMAL(18,2)),
           CAST((SELECT SUM(delivered_orders) FROM fk_ce_2026_07_1372.gold.agg_daily_store_sales) AS DECIMAL(18,2))
    UNION ALL
    -- 5. CURRENT_CUSTOMERS: Distinct customer_id in silver.customers vs distinct customer_id in gold.dim_customer where is_current = TRUE
    SELECT 'CURRENT_CUSTOMERS',
           CAST((SELECT COUNT(DISTINCT customer_id) FROM fk_ce_2026_07_1372.silver.customers) AS DECIMAL(18,2)),
           CAST((SELECT COUNT(DISTINCT customer_id) FROM fk_ce_2026_07_1372.gold.dim_customer WHERE is_current = TRUE) AS DECIMAL(18,2))
) checks;

SELECT * FROM fk_ce_2026_07_1372.ops.reconciliation_results
ORDER BY run_ts DESC, check_name;
